In [ ]:
import os
import json
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# Building conditioning tensor and data splitting

End-to-end pipeline that turns the raw ERA5 / MCD64A1 / static layers into a `(sample, channel, lat, lon)` conditioning tensor and a `(sample, lat, lon)` target tensor, split into train / val / test.

## 1. Loading all the data

### 1.1. Loading ERA5 monthly climate

In [ ]:
# Load the three ERA5 monthly streams.
#   *_temperature_*.nc   -> t2m
#   *_precipitation_*.nc -> tp
#   era5_derived_monthly.nc -> vpd_monthly_mean_daymax, wind_speed_monthly
era5_t   = xr.open_mfdataset("data/era5_monthly/era5_monthly_temperature_*.nc",   combine="by_coords", parallel=True)
era5_p   = xr.open_mfdataset("data/era5_monthly/era5_monthly_precipitation_*.nc", combine="by_coords", parallel=True)
era5_drv = xr.open_mfdataset("data/era5_derived_monthly.nc",                       combine="by_coords", parallel=True)

# Floor timestamps to day so the three streams merge cleanly.
era5_t   = era5_t.assign_coords(valid_time=era5_t.valid_time.dt.floor("D"))
era5_p   = era5_p.assign_coords(valid_time=era5_p.valid_time.dt.floor("D"))
era5_drv = era5_drv.assign_coords(valid_time=era5_drv.valid_time.dt.floor("D"))

ds_monthly = xr.merge([era5_t, era5_p, era5_drv], compat="override")

# Drop CDS scalar coords if present (older ERA5 carries `number`, ERA5T carries `expver`).
for c in ["number", "expver"]:
    if c in ds_monthly.coords:
        ds_monthly = ds_monthly.drop_vars(c)

ds_monthly

In [ ]:
# Load the static land-sea mask and reindex to the climate grid.
lsm = xr.open_dataset("data/era5_static/era5_lsm.nc")["lsm"].squeeze()
lsm = lsm.reindex_like(ds_monthly, method="nearest")

# Land-only view of the climate, used downstream for normalisation statistics
# so that ocean cells do not contaminate the per-channel mean/std.
ds_monthly_land = ds_monthly.where(lsm > 0.5)


### 1.2. Loading MCD64A1 burned area

In [ ]:
burned_area = xr.open_mfdataset(
    "data/mcd64a1/burned_fraction_iberia_0p25deg_monthly.nc",
    combine="by_coords", parallel=True,
)
burned_area

### 1.3. Loading static layers

In [ ]:
fuel_ds         = xr.open_mfdataset("data/processed/static_landcover_fuelfractions.nc", combine="by_coords", parallel=True)
dem_ds          = xr.open_mfdataset("data/processed/static_topography.nc",              combine="by_coords", parallel=True)
static_clim_ds  = xr.open_mfdataset("data/processed/static_ndvi_climatology.nc",        combine="by_coords", parallel=True)
dynamic_clim_ds = xr.open_mfdataset("data/processed/dynamic_ndvi_anomaly.nc",           combine="by_coords", parallel=True)


## 2. Coordinate harmonisation

ERA5 uses `(valid_time, latitude, longitude)` with descending latitude; the burned-area target and the static layers use `(time, lat, lon)` with ascending latitude. Standardise everything to `(time, lat, lon)` ascending in latitude so the rest of the notebook can align by name without surprises.

In [ ]:
def harmonise(ds):
    """Rename ERA5 coords to (time, lat, lon), sort lat ascending, drop ERA5 metadata coords."""
    rename = {}
    for old, new in [("valid_time", "time"), ("latitude", "lat"), ("longitude", "lon")]:
        if old in ds.dims or old in ds.coords:
            rename[old] = new
    if rename:
        ds = ds.rename(rename)
    if "lat" in ds.coords:
        ds = ds.sortby("lat")
    for c in ["expver", "number"]:
        if c in ds.coords:
            ds = ds.drop_vars(c)
    return ds

ds_monthly      = harmonise(ds_monthly)
ds_monthly_land = harmonise(ds_monthly_land)
lsm             = harmonise(lsm)
burned_area     = burned_area.sortby("lat")
fuel_ds         = fuel_ds.sortby("lat")
dem_ds          = dem_ds.sortby("lat")
static_clim_ds  = static_clim_ds.sortby("lat")
dynamic_clim_ds = dynamic_clim_ds.sortby("lat")

print("ds_monthly:  ", dict(ds_monthly.sizes),  "lat", ds_monthly.lat.values[[0, -1]],
      "time", str(ds_monthly.time.values[0])[:10],  "→", str(ds_monthly.time.values[-1])[:10])
print("burned_area:", dict(burned_area.sizes), "lat", burned_area.lat.values[[0, -1]],
      "time", str(burned_area.time.values[0])[:10], "→", str(burned_area.time.values[-1])[:10])


## 3. Train / val / test split

- **Test (60 months):** 2017, 2022, 2023, 2024, 2025 — includes both case-study years (2017 Portugal, 2022 Iberian summer) plus three additional years spanning quiet, moderate, and extreme regimes.
- **Validation (24 months):** 2014, 2019 — temporally distant from each other and from the test years; low- and moderate-activity respectively.
- **Train (~218 months):** all remaining months from Nov 2000 through Dec 2021, excluding the testing and validation years.

The model is non-autoregressive (each sample is `(K months of climate, statics) → one month of burned fraction`, with no past burned fraction in the input), so the apparent non-chronology of the split does not introduce target leakage.

In [ ]:
TEST_YEARS = {2017, 2022, 2023, 2024, 2025}
VAL_YEARS  = {2014, 2019}
TRAIN_END  = 2021  # train spans Nov 2000 .. Dec 2021 minus VAL_YEARS

target_times = pd.DatetimeIndex(burned_area.time.values)

def split_label(t):
    if t.year in TEST_YEARS: return "test"
    if t.year in VAL_YEARS:  return "val"
    if t.year <= TRAIN_END:  return "train"
    return "test"  # never reached for the current dataset

split = pd.Series([split_label(t) for t in target_times],
                  index=target_times, name="split")

print(split.value_counts().to_string())
print(f"total: {len(split)} target months")

train_times = split[split == "train"].index
val_times   = split[split == "val"].index
test_times  = split[split == "test"].index


## 4. Static NDVI climatology — recompute from train years only

The static NDVI climatology is built from train-split years only — restricting the climatology to train years avoids the subtle leakage that would arise from including 2017 and 2022–2025 NDVI in a "static" input used for those test targets. 

In [ ]:
# `dynamic_clim_ds` labels NDVI mid-month (e.g. 2000-11-15); the burned-area target labels
# first-of-month. Snap dynamic NDVI to first-of-month so its time axis is comparable.
ndvi_full = dynamic_clim_ds["ndvi"].assign_coords(
    time=pd.DatetimeIndex(dynamic_clim_ds.time.values).to_period("M").to_timestamp()
)

# Mean NDVI per calendar month, computed over train months only.
train_ndvi = ndvi_full.sel(time=train_times)
ndvi_clim  = train_ndvi.groupby("time.month").mean("time").compute()

# Diagnostic: how much does the train-only climatology differ from the full dataset?
diff = (ndvi_clim - static_clim_ds["ndvi_climatology"]).load()
max_abs_diff = float(xr.apply_ufunc(np.abs, diff.fillna(0)).max())
print(f"Recomputed climatology shape: {ndvi_clim.shape}")
print(f"Train-only vs complete NDVI climatology — max |diff|: {max_abs_diff:.4f}")


## 5. Lagged climate

For each target month `t`, the conditioning needs climate from months `t-K, …, t-1` with `K = 4`. Build one DataArray per `(variable, lag)` pair, indexed by the **target** month — i.e. `clim_lag[("t2m", 1)].sel(time=2017-08-01)` returns the July-2017 t2m field.

In [ ]:
K = 4
CLIM_VARS = ["t2m", "tp", "vpd_monthly_mean_daymax", "wind_speed_monthly"]
SHORT     = {"t2m": "t2m", "tp": "tp",
             "vpd_monthly_mean_daymax": "vpd",
             "wind_speed_monthly":      "wind"}

# Sanity check: ERA5 must cover the earliest lagged month
needed_start = (target_times[0] - pd.DateOffset(months=K)).to_datetime64()
assert ds_monthly.time.min().values <= needed_start, \
    f"ERA5 starts at {ds_monthly.time.min().values}, earliest history needed is {needed_start}"

clim_lag = {}
for k in range(1, K + 1):
    times_minus_k = target_times - pd.DateOffset(months=k)
    for var in CLIM_VARS:
        da = ds_monthly[var].sel(time=times_minus_k)
        # Re-label time coord from t-k -> t so all lags share the target time axis
        da = da.assign_coords(time=target_times.values)
        clim_lag[(var, k)] = da

print(f"Built {len(clim_lag)} lagged climate channels (K={K} × {len(CLIM_VARS)} variables).")


## 6. Normalisation statistics

Per-channel z-score statistics computed over **train** months only, with the LSM applied so ocean cells do not contaminate the moments. Precipitation is `log1p`-transformed before z-scoring (right-skewed and bounded below at zero — naive z-scoring leaves a heavy tail that destabilises early diffusion training). Static fractional channels (LSM, fuel) and the calendar `sin`/`cos` channels are left untransformed.

In [ ]:
ds_train_land = ds_monthly_land.sel(time=train_times)

stats = {}

# --- Climate variables ---
for var in CLIM_VARS:
    da = ds_train_land[var]
    if var == "tp":
        da = np.log1p(da.where(da >= 0, 0))
        transform = "log1p_then_zscore"
    else:
        transform = "zscore"
    stats[var] = {
        "mean":      float(da.mean().compute().values),
        "std":       float(da.std().compute().values),
        "transform": transform,
    }

# --- Topography (z-score over land) ---
for elev_var in ["elev_mean", "elev_std", "slope_mean"]:
    da = dem_ds[elev_var].where(lsm > 0.5)
    stats[elev_var] = {
        "mean":      float(da.mean().compute().values),
        "std":       float(da.std().compute().values),
        "transform": "zscore",
    }
# --- NDVI climatology (z-score over 12 months × land cells) ---
ndvi_clim_land = ndvi_clim.where(lsm > 0.5)
stats["ndvi_clim"] = {
    "mean":      float(ndvi_clim_land.mean().compute().values),
    "std":       float(ndvi_clim_land.std().compute().values),
    "transform": "zscore",
}

# Fuel fractions, LSM, calendar sin/cos: kept as-is (already bounded).
print(json.dumps(stats, indent=2))


## 7. Assemble the conditioning tensor

Channel order (variable-major):

| Group | Channels |
|---|---|
| Dynamic climate (16) | `t2m_lag{4..1}`, `tp_lag{4..1}`, `vpd_lag{4..1}`, `wind_lag{4..1}` |
| Static (12) | `lsm`, `elev_mean`, `elev_std`, `slope_mean`, `fuel_<class>` × 7, `ndvi_clim` |
| Calendar (2) | `month_sin`, `month_cos` |

Final shape: `(time=302, channel=30, lat=32, lon=54)`.

In [ ]:
# --- Helpers ----------------------------------------------------------
def zscore(da, key):
    s = stats[key]
    return (da - s["mean"]) / s["std"]

def log1p_zscore(da, key):
    s = stats[key]
    return (np.log1p(da.where(da >= 0, 0)) - s["mean"]) / s["std"]

ones_t = xr.DataArray(np.ones(len(target_times), dtype="float32"),
                      dims="time", coords={"time": target_times.values})

def to_T(da_2d):
    """Broadcast a (lat, lon) field to (time, lat, lon) along target_times."""
    return (da_2d * ones_t).transpose("time", "lat", "lon").astype("float32")

# --- Build channels ---------------------------------------------------
channels = []

# 1) Dynamic climate, variable-major, lag from K..1
for var in CLIM_VARS:
    short = SHORT[var]
    for k in range(K, 0, -1):
        da = clim_lag[(var, k)]
        da_norm = log1p_zscore(da, var) if var == "tp" else zscore(da, var)
        channels.append((f"{short}_lag{k}", da_norm.astype("float32")))

# 2a) LSM (no normalisation)
channels.append(("lsm", to_T(lsm.astype("float32"))))

# 2b) Topography (z-scored)
for elev_var in ["elev_mean", "elev_std", "slope_mean"]:
    da_norm = zscore(dem_ds[elev_var], elev_var)
    channels.append((elev_var, to_T(da_norm)))

# 2c) Fuel fractions (no normalisation)
for fc in fuel_ds.fuel_class.values:
    fuel_da = fuel_ds["fuel_fraction"].sel(fuel_class=fc).drop_vars("fuel_class")
    channels.append((f"fuel_{fc}", to_T(fuel_da)))

# 2d) NDVI climatology indexed by target month, then z-scored
months_idx = target_times.month.values
months_idx_da = xr.DataArray(months_idx, dims="time", coords={"time": target_times.values})
ndvi_per_t = ndvi_clim.sel(month=months_idx_da)
if "month" in ndvi_per_t.coords:
    ndvi_per_t = ndvi_per_t.drop_vars("month")
ndvi_per_t = zscore(ndvi_per_t, "ndvi_clim").astype("float32")
channels.append(("ndvi_clim", ndvi_per_t))

# 3) Calendar sin/cos (no normalisation)
month_sin = xr.DataArray(np.sin(2 * np.pi * months_idx / 12).astype("float32"),
                         dims="time", coords={"time": target_times.values})
month_cos = xr.DataArray(np.cos(2 * np.pi * months_idx / 12).astype("float32"),
                         dims="time", coords={"time": target_times.values})
ones_2d = xr.ones_like(lsm).astype("float32")
channels.append(("month_sin", (month_sin * ones_2d).transpose("time", "lat", "lon")))
channels.append(("month_cos", (month_cos * ones_2d).transpose("time", "lat", "lon")))

# --- Stack into (time, channel, lat, lon) ----------------------------
channel_names = [name for name, _ in channels]
conditioning = (xr.concat([da for _, da in channels], dim="channel")
                  .assign_coords(channel=channel_names)
                  .transpose("time", "channel", "lat", "lon"))

print(f"Conditioning tensor: shape {conditioning.shape}, dims {conditioning.dims}")
print(f"Channels ({len(channel_names)}):")
for i, n in enumerate(channel_names):
    print(f"  {i:2d}  {n}")


## 8. Target tensor

Burned fraction in `[0, 1]` from MCD64A1, already aggregated to the 0.25° grid. Saved as-is — any `[-1, 1]` rescaling for the diffusion model is a training-time decision, not a dataset-build decision.

In [ ]:
target = burned_area["burned_fraction"].astype("float32")
target = target.assign_coords(time=target_times.values)
assert target.shape == (len(target_times), 32, 54)
print(f"Target tensor: shape {target.shape}, dims {target.dims}")


## 9. Sanity check — visualise one sample

Pick August 2017 (the month of the deadliest Portugal fires) and plot a representative subset of channels alongside the target.

In [ ]:
sample_t = pd.Timestamp("2017-08-01")
to_plot = ["t2m_lag1", "tp_lag1", "vpd_lag1", "wind_lag1",
           "lsm",      "elev_mean", "fuel_forest", "ndvi_clim"]

fig, axes = plt.subplots(2, 4, figsize=(16, 6.5), constrained_layout=True)
for ax, ch in zip(axes.flat, to_plot):
    conditioning.sel(time=sample_t, channel=ch).plot(ax=ax, add_colorbar=True)
    ax.set_title(ch)
fig.suptitle(f"Conditioning channels — target month {sample_t.strftime('%Y-%m')}")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
target.sel(time=sample_t).plot(ax=ax, vmin=0, vmax=0.3)
ax.set_title(f"Target — burned fraction, {sample_t.strftime('%Y-%m')}")
plt.show()


## 10. Save splits and normalisation stats

For each split write a single NetCDF containing the `conditioning` tensor `(sample, channel, lat, lon)` and the `target` tensor `(sample, lat, lon)`, indexed by the target month. The LSM is saved separately (used for masking the loss at training time and for plotting). Normalisation stats are saved as JSON so they can be loaded alongside model weights at inference time — at inference, ERA5 inputs must be transformed using exactly the train-time statistics.

In [ ]:
out_dir = "data/processed/splits"
os.makedirs(out_dir, exist_ok=True)

# Static artefacts
lsm.to_dataset(name="lsm").to_netcdf("data/processed/static_lsm.nc")
ndvi_clim.to_dataset(name="ndvi_climatology_train").to_netcdf("data/processed/static_ndvi_climatology_train.nc")

with open("data/processed/norm_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

# Splits
for split_name, split_idx in [("train", train_times), ("val", val_times), ("test", test_times)]:
    cond_split   = conditioning.sel(time=split_idx).load()
    target_split = target.sel(time=split_idx).load()

    ds_out = xr.Dataset({
        "conditioning": cond_split,
        "target":       target_split,
    })
    ds_out.attrs["split"]         = split_name
    ds_out.attrs["n_samples"]     = int(len(split_idx))
    ds_out.attrs["K"]             = K
    ds_out.attrs["channel_order"] = ", ".join(channel_names)

    out_path = os.path.join(out_dir, f"{split_name}.nc")
    ds_out.to_netcdf(out_path)
    print(f"{split_name:5s}  conditioning {tuple(cond_split.shape)}  target {tuple(target_split.shape)}  →  {out_path}")


## 11. Summary

Outputs written to `data/processed/`:

- `splits/train.nc`, `splits/val.nc`, `splits/test.nc` — each with `conditioning(time, channel, lat, lon)`, `target(time, lat, lon)`, and a `channel` coord listing the channel names in order.
- `static_lsm.nc` — land-sea mask, used at training time to mask the loss to land cells and at plotting time.
- `static_ndvi_climatology_train.nc` — the train-only NDVI climatology used to build the conditioning tensor.
- `norm_stats.json` — per-channel mean/std and transform tag, to be loaded alongside model weights at inference time.

**Channel layout (30 channels)**

| Group | Channels |
|---|---|
| Dynamic climate (16) | `t2m_lag{4..1}`, `tp_lag{4..1}`, `vpd_lag{4..1}`, `wind_lag{4..1}` |
| Static (12) | `lsm`, `elev_mean`, `elev_std`, `slope_mean`, `fuel_<class>` × 7, `ndvi_clim` |
| Calendar (2) | `month_sin`, `month_cos` |

Plus a single-channel `target` (burned fraction in `[0, 1]`).

**Split sizes**: train ≈ 218 months, val = 24, test = 60 — totalling 302 monthly samples covering Nov 2000 – Dec 2025.